# Lab Manual — NumPy Arrays & pandas

**Scientific Computing / Data Handling Laboratory**

---

| | |
|---|---|
| **Student name** | _________________________ |
| **Roll / ID no.** | _________________________ |
| **Batch / Section** | _________________________ |
| **Date of experiment** | _________________________ |
| **Date of submission** | _________________________ |

---

## How to use this manual

1. Run every cell **in order** — later cells depend on variables created earlier.
   Use `Shift + Enter` to run a cell and move to the next one.
2. Read the *Theory* block before running the demo cells.
3. Cells marked **`# TASK`** are for you to fill in. Replace `...` with your code.
4. Solutions are hidden at the end of each lab. Try first, peek later.
5. At the end of each lab, answer the **Viva questions** in the markdown cell provided
   (double-click the cell to edit it).

> **Tip (Colab):** `Runtime → Restart and run all` re-runs the whole notebook from a clean
> state. Do this once before you submit, to make sure your notebook runs top to bottom
> without errors.

---

## Contents

| Lab | Title | Topics |
|---|---|---|
| 0 | Setting up the environment | versions, imports, display options |
| 1 | Creating NumPy arrays | `array`, `arange`, `linspace`, `zeros`, `ones`, `eye`, `random` |
| 2 | Array attributes, dtypes & reshaping | `ndim`, `shape`, `dtype`, `reshape`, `ravel`, `T` |
| 3 | Indexing, slicing & boolean masks | basic/fancy indexing, views vs copies |
| 4 | Vectorised arithmetic & broadcasting | ufuncs, broadcasting rules |
| 5 | Aggregation, sorting & linear algebra | `sum`, `mean`, `axis=`, `argsort`, `@`, `linalg` |
| 6 | pandas Series | creation, index, vectorised ops |
| 7 | pandas DataFrame — building & inspecting | construction, `head`, `info`, `describe` |
| 8 | Selection & filtering | `loc`, `iloc`, boolean filters, `query` |
| 9 | Cleaning & missing data | `isna`, `fillna`, `dropna`, `astype`, duplicates |
| 10 | Grouping & aggregation | `groupby`, `agg`, `pivot_table`, `crosstab` |
| 11 | Combining datasets | `merge`, `join`, `concat` |
| 12 | Time series, I/O & plotting | `to_datetime`, `resample`, `read_csv`, `plot` |
| 13 | Mini project — end-to-end analysis | everything combined |

---

# Lab 0 — Setting Up the Environment

### Aim
To verify the Python scientific stack and configure a comfortable working environment.

### Theory
**NumPy** provides the `ndarray`: a fixed-size, homogeneous (single dtype), N-dimensional
block of memory. Because every element has the same type and the data sits in one
contiguous buffer, loops run in compiled C instead of interpreted Python — typically
10–100× faster than a Python list.

**pandas** builds on NumPy and adds *labels*. A `Series` is a 1-D array with an index;
a `DataFrame` is a table of Series sharing one index. Labels let you align, join and
group data without tracking positions by hand.

Google Colab ships with both pre-installed, so no `pip install` is needed.

In [ ]:
import sys
import numpy as np
import pandas as pd

print('Python :', sys.version.split()[0])
print('NumPy  :', np.__version__)
print('pandas :', pd.__version__)

Record the versions in your lab record. Now set a few display options so that wide
tables and floats print nicely.

In [ ]:
# Display settings for the whole notebook
pd.set_option('display.max_columns', 50)      # show up to 50 columns
pd.set_option('display.width', 120)           # wider console output
pd.set_option('display.precision', 3)         # 3 decimal places
np.set_printoptions(precision=3, suppress=True)  # no scientific notation

# A fixed seed makes random results reproducible -> your output matches the manual
rng = np.random.default_rng(seed=42)
print('Environment configured.')

### Observation 0.1 — why arrays instead of lists?
Run the timing comparison below and note the speed-up in your record.

In [ ]:
import time

N = 1_000_000
py_list = list(range(N))
np_arr  = np.arange(N)

t0 = time.perf_counter()
sq_list = [x * x for x in py_list]          # pure Python loop
t1 = time.perf_counter()

t2 = time.perf_counter()
sq_arr = np_arr ** 2                        # vectorised, runs in C
t3 = time.perf_counter()

print(f'Python list : {t1 - t0:.4f} s')
print(f'NumPy array : {t3 - t2:.4f} s')
print(f'Speed-up    : {(t1 - t0) / (t3 - t2):.1f}x')

**Record:** speed-up observed = ____________ ×

---

# Lab 1 — Creating NumPy Arrays

### Aim
To create arrays using every standard constructor and understand when each is appropriate.

### Theory
Arrays can be built four ways:

1. **From existing data** — `np.array(list_or_nested_list)`
2. **From a numeric range** — `np.arange(start, stop, step)` (stop *excluded*),
   `np.linspace(start, stop, num)` (stop *included* by default)
3. **Pre-filled of a given shape** — `np.zeros`, `np.ones`, `np.full`, `np.empty`, `np.eye`
4. **Random** — via a `Generator` from `np.random.default_rng()`

> `np.empty` does *not* zero the memory; it returns whatever bytes were already there.
> Use it only when you will immediately overwrite every element.

In [ ]:
# --- 1. From existing Python data ---
a1 = np.array([1, 2, 3, 4, 5])                      # 1-D
a2 = np.array([[1, 2, 3], [4, 5, 6]])               # 2-D (2 rows x 3 cols)
a3 = np.array([1, 2, 3], dtype=np.float64)          # force a dtype

print('a1 =', a1)
print('a2 =\n', a2)
print('a3 =', a3, '| dtype:', a3.dtype)

In [ ]:
# --- 2. From numeric ranges ---
r1 = np.arange(10)                 # 0..9
r2 = np.arange(2, 20, 3)           # start, stop(excl), step
r3 = np.linspace(0, 1, 5)          # 5 evenly spaced values, endpoint included
r4 = np.linspace(0, 1, 5, endpoint=False)

print('arange(10)            :', r1)
print('arange(2, 20, 3)      :', r2)
print('linspace(0, 1, 5)     :', r3)
print('linspace .. no endpt  :', r4)

In [ ]:
# --- 3. Pre-filled arrays ---
print('zeros(3, 4):\n', np.zeros((3, 4)))
print('\nones(2, 3) :\n', np.ones((2, 3), dtype=int))
print('\nfull(2, 2) :\n', np.full((2, 2), 7.5))
print('\neye(3)     :\n', np.eye(3))              # identity matrix
print('\nzeros_like(a2):\n', np.zeros_like(a2))   # same shape & dtype as a2

In [ ]:
# --- 4. Random arrays (modern Generator API) ---
rng = np.random.default_rng(seed=42)

print('uniform [0,1)  :\n', rng.random((2, 3)))
print('\nintegers 1..6  :\n', rng.integers(low=1, high=7, size=(2, 5)))   # dice rolls
print('\nnormal mu=0,s=1:\n', rng.normal(loc=0, scale=1, size=5))
print('\nshuffled       :', rng.permutation(np.arange(10)))

### Tasks — Lab 1
Complete each `# TASK` cell. Do not delete the `print` lines; they check your answer.

In [ ]:
# TASK 1.1  Create a 1-D array of the first 15 even numbers starting at 2.
even15 = ...

print(even15)
assert even15.shape == (15,) and even15[0] == 2 and even15[-1] == 30

In [ ]:
# TASK 1.2  Create a 4x4 matrix filled with the value 9, of integer dtype.
nines = ...

print(nines)
assert nines.shape == (4, 4) and nines.dtype.kind == 'i' and (nines == 9).all()

In [ ]:
# TASK 1.3  Create 50 evenly spaced values between -pi and pi (both endpoints included).
angles = ...

print(angles[:5], '...', angles[-3:])
assert angles.size == 50 and np.isclose(angles[0], -np.pi) and np.isclose(angles[-1], np.pi)

In [ ]:
# TASK 1.4  Using a generator seeded with 7, draw a 3x3 array of integers in [10, 99].
rng7 = ...
draw = ...

print(draw)
assert draw.shape == (3, 3) and draw.min() >= 10 and draw.max() <= 99

<details>
<summary><b>Solutions — Lab 1</b> (click to expand)</summary>

```python
# 1.1
even15 = np.arange(2, 31, 2)

# 1.2
nines = np.full((4, 4), 9, dtype=int)

# 1.3
angles = np.linspace(-np.pi, np.pi, 50)

# 1.4
rng7 = np.random.default_rng(7)
draw = rng7.integers(10, 100, size=(3, 3))
```
</details>

---

# Lab 2 — Array Attributes, Data Types & Reshaping

### Aim
To inspect array metadata and change an array's shape without copying its data.

### Theory
Every `ndarray` carries metadata separate from its data buffer:

| Attribute | Meaning |
|---|---|
| `ndim` | number of dimensions (axes) |
| `shape` | tuple giving the length of each axis |
| `size` | total number of elements = product of `shape` |
| `dtype` | element type (`int64`, `float64`, `bool`, `<U5` …) |
| `itemsize` | bytes per element |
| `nbytes` | total bytes = `size × itemsize` |

**Reshaping** changes only the metadata, not the buffer — so `reshape` returns a *view*
and is essentially free. One axis may be given as `-1`, meaning "infer this length".

**Upcasting:** mixing dtypes promotes to the most general one
(`int` + `float` → `float`; anything + `str` → `str`).

In [ ]:
arr = np.arange(24).reshape(2, 3, 4)   # a 3-D array: 2 blocks, 3 rows, 4 cols

print('array:\n', arr)
print('\nndim     :', arr.ndim)
print('shape    :', arr.shape)
print('size     :', arr.size)
print('dtype    :', arr.dtype)
print('itemsize :', arr.itemsize, 'bytes')
print('nbytes   :', arr.nbytes, 'bytes')

In [ ]:
# --- dtypes and conversion ---
i = np.array([1, 2, 3])
f = np.array([1.5, 2.5, 3.5])
b = np.array([True, False, True])
s = np.array(['cat', 'dog', 'hamster'])

for name, x in [('int', i), ('float', f), ('bool', b), ('str', s)]:
    print(f'{name:6s} -> dtype {str(x.dtype):8s} itemsize {x.itemsize}')

print('\nupcasting int+float :', (i + f).dtype)
print('astype(int) truncates:', f.astype(int))     # 1.5 -> 1, NOT rounded
print('np.round then cast   :', np.round(f).astype(int))

In [ ]:
# --- reshaping ---
v = np.arange(12)
print('original  :', v)
print('3x4       :\n', v.reshape(3, 4))
print('4x3       :\n', v.reshape(4, 3))
print('2x-1 (inferred):\n', v.reshape(2, -1))

m = v.reshape(3, 4)
print('\ntranspose .T:\n', m.T)
print('\nravel (flatten to 1-D, a view) :', m.ravel())
print('flatten (always a copy)        :', m.flatten())

In [ ]:
# --- adding and removing axes ---
x = np.array([1, 2, 3])
print('x shape          :', x.shape)
print('x[:, None] shape :', x[:, None].shape)   # column vector (3,1)
print('x[None, :] shape :', x[None, :].shape)   # row vector    (1,3)
print('np.squeeze       :', np.squeeze(x[:, None]).shape)  # drops length-1 axes

# stacking
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print('\nvstack:\n', np.vstack([a, b]))
print('hstack:', np.hstack([a, b]))
print('column_stack:\n', np.column_stack([a, b]))

### Observation 2.1 — views share memory
This is the single most common source of bugs for beginners. Run and study it.

In [ ]:
base = np.arange(6)
view = base.reshape(2, 3)     # a VIEW - shares the same buffer
copy = base.reshape(2, 3).copy()  # an independent COPY

view[0, 0] = 999
print('after editing the view, base =', base)   # base changed!

copy[0, 1] = -1
print('after editing the copy, base =', base)   # base unchanged
print('\nview shares memory with base?', np.shares_memory(base, view))
print('copy shares memory with base?', np.shares_memory(base, copy))

### Tasks — Lab 2

In [ ]:
# TASK 2.1  Build a 5x5 matrix holding the numbers 1..25 in row-major order.
mat = ...

print(mat)
assert mat.shape == (5, 5) and mat[0, 0] == 1 and mat[4, 4] == 25

In [ ]:
# TASK 2.2  Convert the float array below to int WITHOUT truncating (round to nearest).
vals = np.array([1.2, 2.7, 3.5, 4.49, 5.51])
rounded = ...

print(rounded)
assert list(rounded) == [1, 3, 4, 4, 6]   # note: NumPy uses banker's rounding on .5

In [ ]:
# TASK 2.3  Reshape `mat` from Task 2.1 into a 1-D array and report its nbytes.
flat = ...
print(flat, '| nbytes =', ...)

<details>
<summary><b>Solutions — Lab 2</b></summary>

```python
# 2.1
mat = np.arange(1, 26).reshape(5, 5)

# 2.2
rounded = np.round(vals).astype(int)

# 2.3
flat = mat.ravel()
print(flat, '| nbytes =', flat.nbytes)
```
</details>

---

# Lab 3 — Indexing, Slicing and Boolean Masks

### Aim
To extract any sub-part of an array using positional, fancy and boolean indexing.

### Theory
Slice syntax is `start:stop:step`, with `stop` excluded; omitted parts default to
"beginning", "end" and `1`. For N-D arrays, give one slice per axis separated by commas:
`arr[rows, cols]`.

Three indexing styles:

| Style | Example | Returns |
|---|---|---|
| **Basic (slices)** | `a[1:4, ::2]` | a **view** |
| **Fancy (integer arrays)** | `a[[0, 2, 4]]` | a **copy** |
| **Boolean mask** | `a[a > 5]` | a **copy**, always 1-D |

A boolean mask is itself an array of `True`/`False` the same shape as the data.
Combine masks with `&` (and), `|` (or), `~` (not) — **not** the Python keywords
`and`/`or`/`not` — and wrap each condition in parentheses.

In [ ]:
a = np.arange(10)
print('a            :', a)
print('a[3]         :', a[3])
print('a[-1]        :', a[-1])        # last element
print('a[2:7]       :', a[2:7])
print('a[:4]        :', a[:4])
print('a[5:]        :', a[5:])
print('a[::2]       :', a[::2])       # every 2nd
print('a[::-1]      :', a[::-1])      # reversed

In [ ]:
m = np.arange(1, 26).reshape(5, 5)
print('m:\n', m)
print('\nm[2, 3]        (single element) :', m[2, 3])
print('m[1]           (whole row 1)    :', m[1])
print('m[:, 2]        (whole column 2) :', m[:, 2])
print('m[1:4, 1:4]    (sub-block)      :\n', m[1:4, 1:4])
print('m[::2, ::2]    (every other)    :\n', m[::2, ::2])
print('m[-2:, -2:]    (bottom-right)   :\n', m[-2:, -2:])

In [ ]:
# --- fancy indexing: pick arbitrary positions ---
v = np.array([10, 20, 30, 40, 50, 60])
print('v[[0, 3, 5]]      :', v[[0, 3, 5]])
print('v[[5, 5, 0]]      :', v[[5, 5, 0]])    # repeats allowed, order free

rows = np.array([0, 2, 4])
cols = np.array([1, 3, 0])
print('m[rows, cols]     :', m[rows, cols])   # pairs (0,1), (2,3), (4,0)
print('m[rows][:, cols]  :\n', m[rows][:, cols])  # full sub-grid instead

In [ ]:
# --- boolean masking ---
data = np.array([12, 45, 7, 88, 23, 56, 3, 91, 34])

mask = data > 40
print('mask        :', mask)
print('data[mask]  :', data[mask])

print('\nbetween 20 and 60 :', data[(data > 20) & (data < 60)])
print('below 10 or above 80 :', data[(data < 10) | (data > 80)])
print('NOT greater than 40  :', data[~mask])

# assignment through a mask
clipped = data.copy()
clipped[clipped > 50] = 50
print('\nclipped at 50 :', clipped)

# useful helpers
print('np.where(data > 40) -> positions :', np.where(data > 40)[0])
print("np.where(cond, x, y) as if-else  :", np.where(data > 40, 'HIGH', 'low'))
print('count of matches                 :', (data > 40).sum())
print('any / all                        :', (data > 90).any(), (data > 0).all())

### Tasks — Lab 3

In [ ]:
grid = np.arange(1, 37).reshape(6, 6)
print(grid)

# TASK 3.1  Extract the third row.
row3 = ...

# TASK 3.2  Extract the last two columns (all rows).
last2 = ...

# TASK 3.3  Extract the central 2x2 block.
centre = ...

# TASK 3.4  Extract all values divisible by 7.
div7 = ...

# TASK 3.5  Replace every even value with 0 (work on a copy named `no_evens`).
no_evens = ...

for nm, val in [('row3', row3), ('last2', last2), ('centre', centre), ('div7', div7)]:
    print(nm, '->', np.asarray(val).ravel())
print('no_evens:\n', no_evens)

<details>
<summary><b>Solutions — Lab 3</b></summary>

```python
row3     = grid[2]
last2    = grid[:, -2:]
centre   = grid[2:4, 2:4]
div7     = grid[grid % 7 == 0]
no_evens = grid.copy()
no_evens[no_evens % 2 == 0] = 0
```
</details>

---

# Lab 4 — Vectorised Arithmetic and Broadcasting

### Aim
To perform element-wise computation without writing loops, and to understand how NumPy
stretches mismatched shapes.

### Theory
A **ufunc** (universal function) applies an operation element-wise: `a + b`, `np.sqrt(a)`,
`np.exp(a)`. Arithmetic operators are just shorthand for ufuncs.

**Broadcasting** lets arrays of different shapes combine. Compare shapes from the
*trailing* axis backwards; two axes are compatible if they are **equal** or one of them
is **1**. The size-1 axis is then stretched (virtually — no memory is copied).

```
  A      (3, 4)        A      (3, 1)        A      (3,)
  B         (4,)       B         (4,)       B      (4,)
  ---------------      ---------------      ---------------
  ->     (3, 4) OK     ->     (3, 4) OK     ->  ERROR: 3 vs 4
```

In [ ]:
x = np.array([1, 2, 3, 4])
y = np.array([10, 20, 30, 40])

print('x + y  :', x + y)
print('y - x  :', y - x)
print('x * y  :', x * y)          # element-wise, NOT matrix product
print('y / x  :', y / x)
print('y // x :', y // x)         # floor division
print('y % 3  :', y % 3)
print('x ** 2 :', x ** 2)
print('\nscalar broadcast, x * 10 :', x * 10)

In [ ]:
# --- mathematical ufuncs ---
t = np.linspace(0, np.pi, 5)
print('t         :', t)
print('sin(t)    :', np.sin(t))
print('cos(t)    :', np.cos(t))
print('sqrt      :', np.sqrt(np.array([1, 4, 9, 16])))
print('exp       :', np.exp(np.array([0, 1, 2])))
print('log       :', np.log(np.array([1, np.e, np.e**2])))
print('abs       :', np.abs(np.array([-3, -1, 2])))
print('clip(2,8) :', np.clip(np.array([1, 5, 9, 12]), 2, 8))

In [ ]:
# --- broadcasting in action ---
M = np.arange(12).reshape(3, 4)
row = np.array([100, 200, 300, 400])      # shape (4,)   -> broadcast down the rows
col = np.array([[1], [2], [3]])           # shape (3,1)  -> broadcast across columns

print('M:\n', M)
print('\nM + row  (shape (3,4)+(4,)):\n', M + row)
print('\nM * col  (shape (3,4)+(3,1)):\n', M * col)

# classic use: outer product / multiplication table
a = np.arange(1, 6)
table = a[:, None] * a[None, :]
print('\n5x5 multiplication table:\n', table)

In [ ]:
# --- a shape mismatch that cannot broadcast ---
try:
    np.arange(12).reshape(3, 4) + np.array([1, 2, 3])   # (3,4) vs (3,)
except ValueError as e:
    print('ValueError:', e)

print('\nFix it by making the second array a column:')
print(np.arange(12).reshape(3, 4) + np.array([1, 2, 3])[:, None])

### Observation 4.1 — normalising data with broadcasting
A standard preprocessing step: subtract each column's mean and divide by its std.

In [ ]:
rng = np.random.default_rng(0)
raw = rng.normal(loc=[50, 5, 1000], scale=[10, 1, 200], size=(8, 3))

print('raw (first 3 rows):\n', raw[:3])
print('\ncolumn means :', raw.mean(axis=0))
print('column stds  :', raw.std(axis=0))

z = (raw - raw.mean(axis=0)) / raw.std(axis=0)   # broadcasting (8,3) - (3,)
print('\nz-scored means (~0) :', z.mean(axis=0))
print('z-scored stds  (~1) :', z.std(axis=0))

### Tasks — Lab 4

In [ ]:
# TASK 4.1  Convert this array of Celsius temperatures to Fahrenheit  (F = C*9/5 + 32).
celsius = np.array([-40, 0, 15, 25, 37, 100])
fahrenheit = ...
print(fahrenheit)
assert list(fahrenheit) == [-40, 32, 59, 77, 98.6, 212]

In [ ]:
# TASK 4.2  Given side lengths, compute the hypotenuse for each pair (use np.hypot or sqrt).
p = np.array([3, 5, 8, 7])
q = np.array([4, 12, 15, 24])
hyp = ...
print(hyp)
assert list(hyp) == [5, 13, 17, 25]

In [ ]:
# TASK 4.3  Build a 4x4 'distance from origin' grid: entry (i,j) = sqrt(i**2 + j**2).
#           Use broadcasting, no loops.
idx = np.arange(4)
dist = ...
print(np.round(dist, 2))
assert dist.shape == (4, 4) and np.isclose(dist[3, 4 - 1], np.sqrt(18))

<details>
<summary><b>Solutions — Lab 4</b></summary>

```python
fahrenheit = celsius * 9 / 5 + 32
hyp = np.hypot(p, q)                      # or np.sqrt(p**2 + q**2)
dist = np.sqrt(idx[:, None]**2 + idx[None, :]**2)
```
</details>

---

# Lab 5 — Aggregation, Sorting and Linear Algebra

### Aim
To reduce arrays to summary statistics along chosen axes, sort them, and perform
matrix operations.

### Theory
Reductions collapse an axis. The `axis` argument names the axis that **disappears**:

- `axis=0` → collapse rows → one result **per column**
- `axis=1` → collapse columns → one result **per row**
- `axis=None` (default) → collapse everything → a scalar

`argmin` / `argmax` / `argsort` return *positions* rather than values — useful when you
need to index another array with the same ordering.

For matrices, `*` is element-wise but `@` (or `np.matmul`) is the true matrix product.

In [ ]:
d = np.array([[ 4, 12,  7],
              [ 9,  3, 15],
              [ 6, 18,  1],
              [11,  5, 10]])
print('data:\n', d)

print('\nsum  all      :', d.sum())
print('sum  axis=0   :', d.sum(axis=0), '  <- one per column')
print('sum  axis=1   :', d.sum(axis=1), ' <- one per row')
print('\nmean axis=0   :', d.mean(axis=0))
print('min  axis=1   :', d.min(axis=1))
print('max  all      :', d.max())
print('std / var     :', round(d.std(), 3), '/', round(d.var(), 3))
print('median        :', np.median(d))
print('cumulative sum of row 0 :', np.cumsum(d[0]))

In [ ]:
# --- positions, not values ---
scores = np.array([72, 95, 58, 88, 95, 61])
names  = np.array(['Asel', 'Bek', 'Chyn', 'Dina', 'Erlan', 'Farid'])

print('argmax (first highest) :', scores.argmax(), '->', names[scores.argmax()])
print('argmin                 :', scores.argmin(), '->', names[scores.argmin()])

order = scores.argsort()[::-1]            # descending order of positions
print('\nranking:')
for rank, i in enumerate(order, start=1):
    print(f'  {rank}. {names[i]:6s} {scores[i]}')

print('\nnp.sort (values)   :', np.sort(scores))
print('np.unique          :', np.unique(scores))
vals, counts = np.unique(scores, return_counts=True)
print('unique with counts :', dict(zip(vals.tolist(), counts.tolist())))

In [ ]:
# --- linear algebra ---
A = np.array([[2, 1],
              [1, 3]])
B = np.array([[1, 0],
              [4, 2]])

print('A * B  (element-wise):\n', A * B)
print('\nA @ B  (matrix product):\n', A @ B)
print('\nA.T (transpose):\n', A.T)
print('\ndeterminant  :', round(np.linalg.det(A), 4))
print('inverse:\n', np.linalg.inv(A))
print('\nA @ inv(A) = I:\n', np.round(A @ np.linalg.inv(A), 10))

# solving  A x = b
b = np.array([8, 13])
x = np.linalg.solve(A, b)
print('\nsolution of Ax=b :', x)
print('check A@x == b   :', np.allclose(A @ x, b))

eigvals, eigvecs = np.linalg.eig(A)
print('\neigenvalues :', eigvals)

### Tasks — Lab 5
The matrix `marks` below holds the marks of 6 students in 4 subjects.

In [ ]:
rng = np.random.default_rng(2024)
marks = rng.integers(35, 100, size=(6, 4))
students = np.array(['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])
subjects = np.array(['Maths', 'Physics', 'Chem', 'CS'])
print(marks)

# TASK 5.1  Total marks for each student.
student_total = ...

# TASK 5.2  Average mark for each subject.
subject_avg = ...

# TASK 5.3  Name of the student with the highest total.
topper = ...

# TASK 5.4  Name of the subject with the lowest average.
hardest = ...

# TASK 5.5  How many marks in the whole table are below 50?
n_fails = ...

print('totals        :', student_total)
print('subject avgs  :', np.round(subject_avg, 2))
print('topper        :', topper)
print('hardest subj  :', hardest)
print('marks < 50    :', n_fails)

<details>
<summary><b>Solutions — Lab 5</b></summary>

```python
student_total = marks.sum(axis=1)
subject_avg   = marks.mean(axis=0)
topper        = students[marks.sum(axis=1).argmax()]
hardest       = subjects[marks.mean(axis=0).argmin()]
n_fails       = (marks < 50).sum()
```
</details>

### Viva questions — NumPy (Labs 1–5)
Answer in this cell (double-click to edit).

1. How does an `ndarray` differ from a Python list? Give two reasons it is faster.
2. What is the difference between a view and a copy? Which does slicing return?
3. State the broadcasting rule in one sentence.
4. `arr.sum(axis=0)` on a (4, 3) array returns an array of what shape, and why?
5. What is the difference between `a * b` and `a @ b` for 2-D arrays?
6. Why must you write `(a > 1) & (a < 5)` instead of `a > 1 and a < 5`?

**Your answers:**

1. 
2. 
3. 
4. 
5. 
6. 

---

# Lab 6 — pandas Series

### Aim
To create and operate on a labelled 1-D array.

### Theory
A `Series` = **values** (a NumPy array) + **index** (the labels) + an optional **name**.
The index is what distinguishes it from a plain array: arithmetic between two Series
**aligns on the index**, not on position, and missing matches become `NaN`.

A Series can be built from a list, a dict (keys become the index), or a scalar.

In [ ]:
# --- creating Series ---
s1 = pd.Series([10, 20, 30, 40])                       # default RangeIndex 0..3
s2 = pd.Series([10, 20, 30, 40], index=['a','b','c','d'])
s3 = pd.Series({'Bishkek': 1_074_000, 'Osh': 322_000, 'Jalal-Abad': 123_000},
               name='population')

print('s1:\n', s1, sep='')
print('\ns2:\n', s2, sep='')
print('\ns3:\n', s3, sep='')
print('\nvalues :', s3.values)       # the underlying NumPy array
print('index  :', list(s3.index))
print('dtype  :', s3.dtype, '| name:', s3.name)

In [ ]:
# --- accessing elements ---
print('by label  s2[\'b\']   :', s2['b'])
print('by label  s2.loc[\'b\']:', s2.loc['b'])
print('by position s2.iloc[1]:', s2.iloc[1])
print('\nslice by label (INCLUSIVE of the end):\n', s2.loc['b':'d'], sep='')
print('\nslice by position (exclusive):\n', s2.iloc[1:3], sep='')
print('\nboolean mask:\n', s2[s2 > 15], sep='')

In [ ]:
# --- vectorised operations, and index alignment ---
q1 = pd.Series({'pens': 120, 'books': 80, 'bags': 45})
q2 = pd.Series({'books': 95, 'pens': 110, 'shoes': 60})

print('q1 * 2:\n', q1 * 2, sep='')
print('\nq1 + q2  (aligned by label; unmatched -> NaN):\n', q1 + q2, sep='')
print('\nadd with fill_value=0:\n', q1.add(q2, fill_value=0), sep='')

In [ ]:
# --- common Series methods ---
marks = pd.Series([78, 91, 64, 91, 55, 88, 73],
                  index=['Asel','Bek','Chyn','Dina','Erlan','Farid','Gulnaz'],
                  name='marks')

print(marks.describe())
print('\nsum/mean/median :', marks.sum(), round(marks.mean(), 2), marks.median())
print('idxmax (top student) :', marks.idxmax())
print('nlargest(3):\n', marks.nlargest(3), sep='')
print('\nsorted by value:\n', marks.sort_values(ascending=False).head(3), sep='')
print('\nvalue_counts:\n', marks.value_counts().head(3), sep='')
print('\napply a function:\n', marks.apply(lambda m: 'pass' if m >= 60 else 'fail'), sep='')

### Tasks — Lab 6

In [ ]:
rain = pd.Series([12, 0, 35, 8, 0, 47, 21],
                 index=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                 name='rainfall_mm')

# TASK 6.1  Total rainfall for the week.
total = ...

# TASK 6.2  The days with no rain (return the index labels as a list).
dry_days = ...

# TASK 6.3  The wettest day's label.
wettest = ...

# TASK 6.4  A new Series of rainfall in centimetres.
rain_cm = ...

print('total   :', total)
print('dry     :', dry_days)
print('wettest :', wettest)
print('cm      :\n', rain_cm, sep='')

<details>
<summary><b>Solutions — Lab 6</b></summary>

```python
total    = rain.sum()
dry_days = rain[rain == 0].index.tolist()
wettest  = rain.idxmax()
rain_cm  = rain / 10
```
</details>

---

# Lab 7 — The DataFrame: Building and Inspecting

### Aim
To construct a DataFrame and run the standard first-look inspection on it.

### Theory
A `DataFrame` is a 2-D table: a dict of Series that share one row index. Columns may
each have their own dtype — that is the key difference from a NumPy 2-D array.

The standard inspection routine for any new dataset:

| Call | Tells you |
|---|---|
| `df.shape` | rows × columns |
| `df.head()` / `df.tail()` | what the data actually looks like |
| `df.info()` | dtypes, non-null counts, memory |
| `df.describe()` | numeric summary statistics |
| `df.columns`, `df.dtypes` | column names and types |
| `df.isna().sum()` | missing values per column |
| `df['col'].value_counts()` | category frequencies |

Always do this *before* any analysis.

In [ ]:
# --- construction from a dict of lists (most common) ---
df = pd.DataFrame({
    'name'   : ['Asel','Bek','Chyn','Dina','Erlan','Farid','Gulnaz','Hakan'],
    'age'    : [20, 22, 21, 23, 20, 24, 22, 21],
    'city'   : ['Bishkek','Osh','Bishkek','Naryn','Osh','Bishkek','Naryn','Osh'],
    'dept'   : ['CS','EE','CS','ME','CS','EE','CS','ME'],
    'marks'  : [78, 91, 64, 88, 55, 91, 73, 69],
    'fees'   : [45000, 52000, 45000, 38000, 45000, 52000, 45000, 38000],
})
df

In [ ]:
# other ways to build the same kind of object
from_records = pd.DataFrame([('Asel', 20), ('Bek', 22)], columns=['name', 'age'])
from_array   = pd.DataFrame(np.arange(6).reshape(2, 3), columns=['x', 'y', 'z'])

print('from list of tuples:\n', from_records, sep='')
print('\nfrom a NumPy array:\n', from_array, sep='')

In [ ]:
# --- the inspection routine ---
print('shape   :', df.shape)
print('columns :', list(df.columns))
print('\ndtypes:\n', df.dtypes, sep='')
print('\nhead(3):')
display(df.head(3))
print('tail(2):')
display(df.tail(2))

In [ ]:
df.info()

In [ ]:
print('numeric summary:')
display(df.describe())

print('\nincluding object columns:')
display(df.describe(include='all'))

In [ ]:
print('missing values per column:\n', df.isna().sum(), sep='')
print('\ncity counts:\n', df['city'].value_counts(), sep='')
print('\ndept proportions:\n', df['dept'].value_counts(normalize=True).round(3), sep='')
print('\nunique cities :', df['city'].unique(), '| nunique:', df['city'].nunique())

In [ ]:
# --- adding, renaming and dropping columns ---
df2 = df.copy()
df2['grade'] = np.where(df2['marks'] >= 85, 'A',
               np.where(df2['marks'] >= 70, 'B', 'C'))
df2['marks_pct'] = (df2['marks'] / 100).round(2)
df2 = df2.rename(columns={'fees': 'annual_fees'})

display(df2)

df3 = df2.drop(columns=['marks_pct'])
print('after drop:', list(df3.columns))

### Tasks — Lab 7

In [ ]:
# TASK 7.1  Build a DataFrame `products` with these columns:
#   item  : Laptop, Mouse, Monitor, Keyboard, Webcam
#   price : 48000, 850, 12500, 2200, 3400
#   qty   : 3, 25, 7, 18, 12
products = ...

display(products)
assert products.shape == (5, 3)

In [ ]:
# TASK 7.2  Add a column `total_value` = price * qty.
...

# TASK 7.3  Print the shape, the dtypes, and the total inventory value.
...

display(products)

<details>
<summary><b>Solutions — Lab 7</b></summary>

```python
products = pd.DataFrame({
    'item' : ['Laptop','Mouse','Monitor','Keyboard','Webcam'],
    'price': [48000, 850, 12500, 2200, 3400],
    'qty'  : [3, 25, 7, 18, 12],
})
products['total_value'] = products['price'] * products['qty']
print(products.shape)
print(products.dtypes)
print('inventory value:', products['total_value'].sum())
```
</details>

---

# Lab 8 — Selection and Filtering

### Aim
To select any subset of rows and columns, by label, by position, and by condition.

### Theory
Two accessors, and you should use one of them explicitly rather than bare `[]`:

| | `.loc[rows, cols]` | `.iloc[rows, cols]` |
| --- | --- | --- |
| addresses by | **label** | **integer position** |
| slice end | **included** | excluded |
| accepts a boolean mask | yes | no (use `.values`) |

Bare `df['col']` selects a *column*; bare `df[mask]` selects *rows*. That inconsistency
is why `.loc` / `.iloc` are preferred.

Combine conditions with `&`, `|`, `~` and parentheses, exactly as with NumPy masks.

In [ ]:
df = pd.DataFrame({
    'name'  : ['Asel','Bek','Chyn','Dina','Erlan','Farid','Gulnaz','Hakan'],
    'age'   : [20, 22, 21, 23, 20, 24, 22, 21],
    'city'  : ['Bishkek','Osh','Bishkek','Naryn','Osh','Bishkek','Naryn','Osh'],
    'dept'  : ['CS','EE','CS','ME','CS','EE','CS','ME'],
    'marks' : [78, 91, 64, 88, 55, 91, 73, 69],
    'fees'  : [45000, 52000, 45000, 38000, 45000, 52000, 45000, 38000],
}).set_index('name')        # make `name` the row label
df

In [ ]:
# --- selecting columns ---
print('one column -> Series:\n', df['marks'].head(3), sep='')
print('\none column as DataFrame:')
display(df[['marks']].head(3))
print('several columns:')
display(df[['dept', 'marks']].head(3))

In [ ]:
# --- .loc : by label ---
print('single row:\n', df.loc['Dina'], sep='')
print('\nrow + column  :', df.loc['Dina', 'marks'])
print('\nlist of rows:')
display(df.loc[['Asel', 'Erlan'], ['city', 'marks']])
print('label slice (Chyn..Farid, INCLUSIVE):')
display(df.loc['Chyn':'Farid', 'city':'marks'])

In [ ]:
# --- .iloc : by position ---
print('first row:\n', df.iloc[0], sep='')
print('\nrows 1-3, cols 0-1 (end excluded):')
display(df.iloc[1:4, 0:2])
print('last two rows, last column:')
display(df.iloc[-2:, -1:])
print('scattered positions:')
display(df.iloc[[0, 3, 5], [1, 3]])

In [ ]:
# --- boolean filtering ---
print('marks above 75:')
display(df[df['marks'] > 75])

print('CS students from Bishkek:')
display(df[(df['dept'] == 'CS') & (df['city'] == 'Bishkek')])

print('Osh or Naryn:')
display(df[df['city'].isin(['Osh', 'Naryn'])])

print('NOT in CS:')
display(df[~(df['dept'] == 'CS')])

print('marks between 65 and 90:')
display(df[df['marks'].between(65, 90)])

In [ ]:
# --- .query(): the same filters, written as a string ---
print("query('marks > 75 and city == \"Bishkek\"'):")
display(df.query('marks > 75 and city == "Bishkek"'))

threshold = 80
print('using a Python variable with @:')
display(df.query('marks >= @threshold'))

In [ ]:
# --- sorting ---
print('by marks, descending:')
display(df.sort_values('marks', ascending=False))

print('by dept then marks:')
display(df.sort_values(['dept', 'marks'], ascending=[True, False]))

print('by index label:')
display(df.sort_index().head(3))

print('top 3 by marks:')
display(df.nlargest(3, 'marks'))

### Observation 8.1 — chained assignment
`df[df.marks > 80]['fees'] = 0` may silently fail, because the left side is a temporary
copy. Always assign through a **single** `.loc` call.

In [ ]:
tmp = df.copy()
tmp.loc[tmp['marks'] > 85, 'fees'] = 0        # correct: one .loc, rows and column together
display(tmp)

### Tasks — Lab 8
Use the `df` defined in this lab.

In [ ]:
# TASK 8.1  Select the `city` and `marks` columns for Bek and Gulnaz, using .loc.
t1 = ...

# TASK 8.2  Select the first three rows and the first two columns, using .iloc.
t2 = ...

# TASK 8.3  All students aged 21 or under whose marks are at least 60.
t3 = ...

# TASK 8.4  All EE and ME students, sorted by marks descending.
t4 = ...

# TASK 8.5  The name of the student paying the highest fees.
t5 = ...

for nm, obj in [('8.1', t1), ('8.2', t2), ('8.3', t3), ('8.4', t4)]:
    print('--- TASK', nm); display(obj)
print('8.5 ->', t5)

<details>
<summary><b>Solutions — Lab 8</b></summary>

```python
t1 = df.loc[['Bek', 'Gulnaz'], ['city', 'marks']]
t2 = df.iloc[0:3, 0:2]
t3 = df[(df['age'] <= 21) & (df['marks'] >= 60)]
t4 = df[df['dept'].isin(['EE', 'ME'])].sort_values('marks', ascending=False)
t5 = df['fees'].idxmax()
```
</details>

---

# Lab 9 — Cleaning and Missing Data

### Aim
To detect and handle missing values, fix wrong dtypes, and remove duplicates.

### Theory
pandas marks missing values as `NaN` (a float) or `NA`. Important consequences:

- `NaN != NaN`, so never test with `== np.nan`; use `.isna()` / `.notna()`.
- An integer column containing a missing value is upcast to `float64`
  (unless you use the nullable `Int64` dtype).
- Arithmetic **skips** `NaN` by default (`mean()` ignores it), but `sum()` of an
  all-NaN column is `0`.

Handling strategies: **drop** (`dropna`), **fill** with a constant / mean / median /
mode (`fillna`), or **interpolate**. Which is right depends on the data — state your
reasoning in the lab record.

In [ ]:
messy = pd.DataFrame({
    'id'      : [1, 2, 3, 4, 5, 6, 6],
    'name'    : ['Asel', 'Bek', None, 'Dina', 'Erlan', 'Farid', 'Farid'],
    'age'     : [20, np.nan, 21, 23, 20, 24, 24],
    'city'    : ['bishkek', 'OSH', 'Bishkek ', None, 'osh', 'Bishkek', 'Bishkek'],
    'marks'   : ['78', '91', '64', 'absent', '55', '91', '91'],
    'joined'  : ['2024-09-01','2024-09-03','2024-09-01','2024-10-15','2024-09-03',
                 '2024-11-02','2024-11-02'],
})
messy

In [ ]:
# --- diagnose ---
messy.info()
print('\nmissing per column:\n', messy.isna().sum(), sep='')
print('\ntotal missing cells :', messy.isna().sum().sum())
print('rows with any missing:', messy.isna().any(axis=1).sum())
print('duplicate rows       :', messy.duplicated().sum())

In [ ]:
clean = messy.copy()

# 1. drop exact duplicate rows
clean = clean.drop_duplicates()
print('after drop_duplicates:', clean.shape)

# 2. fix the `marks` column: 'absent' is not a number -> coerce to NaN, then numeric
clean['marks'] = pd.to_numeric(clean['marks'], errors='coerce')
print('\nmarks dtype now:', clean['marks'].dtype)

# 3. parse dates
clean['joined'] = pd.to_datetime(clean['joined'])

# 4. normalise the text column: strip whitespace, title case
clean['city'] = clean['city'].str.strip().str.title()

display(clean)

In [ ]:
# --- filling missing values ---
filled = clean.copy()

filled['age']   = filled['age'].fillna(filled['age'].median())      # median: robust
filled['marks'] = filled['marks'].fillna(filled['marks'].mean().round(1))
filled['name']  = filled['name'].fillna('Unknown')
filled['city']  = filled['city'].fillna(filled['city'].mode()[0])   # most common

display(filled)
print('remaining missing:', filled.isna().sum().sum())

In [ ]:
# --- the alternative: dropping ---
print('dropna() any  :', clean.dropna().shape)
print('dropna(how="all") :', clean.dropna(how='all').shape)
print('dropna(subset=["marks"]) :', clean.dropna(subset=['marks']).shape)
print('dropna(thresh=5) keep rows with >=5 non-null :', clean.dropna(thresh=5).shape)

# forward / backward fill and interpolation (useful for ordered data)
series = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print('\noriginal     :', series.tolist())
print('ffill        :', series.ffill().tolist())
print('bfill        :', series.bfill().tolist())
print('interpolate  :', series.interpolate().tolist())

In [ ]:
# --- string cleaning toolkit ---
txt = pd.Series(['  Hello World ', 'PANDAS rocks', 'data_science_101', None])
print('strip      :', txt.str.strip().tolist())
print('lower      :', txt.str.lower().tolist())
print('replace    :', txt.str.replace('_', ' ').tolist())
print('contains   :', txt.str.contains('pandas', case=False, na=False).tolist())
print('split [0]  :', txt.str.split().str[0].tolist())
print('len        :', txt.str.len().tolist())

### Tasks — Lab 9

In [ ]:
raw = pd.DataFrame({
    'product' : [' laptop', 'MOUSE', 'monitor ', 'keyboard', 'mouse', None],
    'price'   : ['48000', '850', 'N/A', '2200', '850', '3400'],
    'qty'     : [3, 25, 7, np.nan, 25, 12],
})
display(raw)

# TASK 9.1  Strip whitespace and convert `product` to lower case.
...

# TASK 9.2  Convert `price` to a numeric column ('N/A' must become NaN).
...

# TASK 9.3  Fill missing `price` with the column median and missing `qty` with 0.
...

# TASK 9.4  Drop rows where `product` is missing, then drop duplicate rows.
...

display(raw)
print(raw.dtypes)

<details>
<summary><b>Solutions — Lab 9</b></summary>

```python
raw['product'] = raw['product'].str.strip().str.lower()
raw['price']   = pd.to_numeric(raw['price'], errors='coerce')
raw['price']   = raw['price'].fillna(raw['price'].median())
raw['qty']     = raw['qty'].fillna(0)
raw = raw.dropna(subset=['product']).drop_duplicates()
```
</details>

---

# Lab 10 — Grouping and Aggregation

### Aim
To compute statistics per category using the split–apply–combine pattern.

### Theory
`groupby` works in three stages:

1. **Split** the rows into groups by one or more key columns
2. **Apply** a function to each group (`mean`, `sum`, `count`, a custom function…)
3. **Combine** the results into a new Series or DataFrame

`df.groupby('k')` by itself produces a lazy object — nothing is computed until you
call an aggregation on it.

`pivot_table` is `groupby` plus a reshape: one grouping key becomes the rows, another
becomes the columns.

In [ ]:
sales = pd.DataFrame({
    'region'  : ['North','South','North','East','South','East','North','South','East','North'],
    'rep'     : ['Asel','Bek','Chyn','Dina','Bek','Dina','Asel','Erlan','Dina','Chyn'],
    'product' : ['Laptop','Mouse','Laptop','Monitor','Laptop','Mouse','Monitor','Monitor','Laptop','Mouse'],
    'units'   : [5, 40, 3, 12, 7, 33, 9, 15, 4, 28],
    'revenue' : [240000, 34000, 144000, 150000, 336000, 28000, 112000, 187500, 192000, 23800],
    'quarter' : ['Q1','Q1','Q1','Q2','Q2','Q2','Q3','Q3','Q3','Q4'],
})
sales

In [ ]:
# --- single-key grouping ---
print('total revenue per region:\n', sales.groupby('region')['revenue'].sum(), sep='')
print('\nmean units per product:\n', sales.groupby('product')['units'].mean().round(1), sep='')
print('\nnumber of sales per rep:\n', sales.groupby('rep').size(), sep='')

print('\nseveral statistics at once:')
display(sales.groupby('region')['revenue'].agg(['count', 'sum', 'mean', 'max']).round(0))

In [ ]:
# --- multiple keys, multiple aggregations ---
print('by region AND product:')
display(sales.groupby(['region', 'product'])['revenue'].sum())

print('\ndifferent function per column:')
display(sales.groupby('region').agg(
    n_sales     = ('revenue', 'count'),
    total_rev   = ('revenue', 'sum'),
    avg_rev     = ('revenue', 'mean'),
    total_units = ('units', 'sum'),
    top_rep     = ('rep', lambda s: s.mode()[0]),
).round(0))

In [ ]:
# --- transform: broadcast a group statistic back to every row ---
sales2 = sales.copy()
sales2['region_total'] = sales2.groupby('region')['revenue'].transform('sum')
sales2['share_of_region'] = (sales2['revenue'] / sales2['region_total'] * 100).round(1)
display(sales2[['region', 'rep', 'revenue', 'region_total', 'share_of_region']])

# filter: keep only groups satisfying a condition
big = sales.groupby('region').filter(lambda g: g['revenue'].sum() > 300000)
print('\nrows from regions with total revenue > 300000:')
display(big)

In [ ]:
# --- pivot_table ---
print('revenue, region x product:')
display(pd.pivot_table(sales, values='revenue', index='region', columns='product',
                       aggfunc='sum', fill_value=0, margins=True, margins_name='TOTAL'))

print('\nunits, quarter x region, mean:')
display(pd.pivot_table(sales, values='units', index='quarter', columns='region',
                       aggfunc='mean', fill_value=0).round(1))

print('\ncrosstab: how many sales of each product in each region?')
display(pd.crosstab(sales['region'], sales['product'], margins=True))

In [ ]:
# --- binning a continuous column, then grouping on the bins ---
sales3 = sales.copy()
sales3['size'] = pd.cut(sales3['revenue'],
                        bins=[0, 50000, 150000, np.inf],
                        labels=['small', 'medium', 'large'])
display(sales3[['rep', 'revenue', 'size']])
print('\ncount per size band:\n', sales3.groupby('size', observed=True).size(), sep='')

### Tasks — Lab 10

In [ ]:
# TASK 10.1  Total units sold per quarter.
a1 = ...

# TASK 10.2  For each rep: number of sales, total revenue, and average units.
a2 = ...

# TASK 10.3  The region with the highest average revenue per sale.
a3 = ...

# TASK 10.4  A pivot table of total units with rows=rep, columns=product, zeros for blanks.
a4 = ...

print('10.1'); display(a1)
print('10.2'); display(a2)
print('10.3 ->', a3)
print('10.4'); display(a4)

<details>
<summary><b>Solutions — Lab 10</b></summary>

```python
a1 = sales.groupby('quarter')['units'].sum()
a2 = sales.groupby('rep').agg(n=('revenue','count'),
                              total=('revenue','sum'),
                              avg_units=('units','mean')).round(1)
a3 = sales.groupby('region')['revenue'].mean().idxmax()
a4 = pd.pivot_table(sales, values='units', index='rep', columns='product',
                    aggfunc='sum', fill_value=0)
```
</details>

---

# Lab 11 — Combining Datasets

### Aim
To join tables on a key column and to stack tables together.

### Theory
`pd.merge(left, right, on='key', how=...)` is a database join:

| `how` | Keeps |
|---|---|
| `'inner'` (default) | only keys present in **both** |
| `'left'` | all rows of the left table |
| `'right'` | all rows of the right table |
| `'outer'` | all keys from both, `NaN` where unmatched |

`pd.concat([a, b])` stacks rows (`axis=0`) or columns (`axis=1`) without matching on a key.

Before merging, always check for duplicate keys: a one-to-many join multiplies rows,
and a many-to-many join can explode your table.

In [ ]:
students = pd.DataFrame({
    'student_id': [1, 2, 3, 4, 5],
    'name'      : ['Asel', 'Bek', 'Chyn', 'Dina', 'Erlan'],
    'dept_id'   : [10, 20, 10, 30, 20],
})

depts = pd.DataFrame({
    'dept_id'  : [10, 20, 30, 40],
    'dept_name': ['Computer Science', 'Electrical', 'Mechanical', 'Civil'],
    'building' : ['A', 'B', 'C', 'D'],
})

scores = pd.DataFrame({
    'student_id': [1, 2, 3, 6],
    'exam_score': [78, 91, 64, 70],
})

print('students:'); display(students)
print('depts:');    display(depts)
print('scores:');   display(scores)

In [ ]:
print('INNER join students x depts (only matching dept_id):')
display(pd.merge(students, depts, on='dept_id', how='inner'))

print('\nLEFT join students x scores (keep all students):')
display(pd.merge(students, scores, on='student_id', how='left'))

print('\nOUTER join students x scores (keep everything):')
display(pd.merge(students, scores, on='student_id', how='outer'))

print('\nindicator=True shows where each row came from:')
display(pd.merge(students, scores, on='student_id', how='outer', indicator=True))

In [ ]:
# --- joining on differently named keys, and chaining merges ---
full = (students
        .merge(depts, on='dept_id', how='left')
        .merge(scores, on='student_id', how='left'))
display(full)

# different column names on each side:
left  = pd.DataFrame({'sid': [1, 2], 'v': ['x', 'y']})
right = pd.DataFrame({'student_id': [1, 2], 'w': ['p', 'q']})
display(pd.merge(left, right, left_on='sid', right_on='student_id'))

In [ ]:
# --- concat: stacking ---
jan = pd.DataFrame({'item': ['A', 'B'], 'sales': [100, 200]})
feb = pd.DataFrame({'item': ['A', 'C'], 'sales': [150, 90]})

print('stack rows:')
display(pd.concat([jan, feb], ignore_index=True))

print('\nstack rows with a label for the source:')
display(pd.concat([jan, feb], keys=['Jan', 'Feb']))

print('\nstack columns (axis=1):')
display(pd.concat([jan, feb.rename(columns={'sales': 'sales_feb'})], axis=1))

### Observation 11.1 — duplicate keys multiply rows
Check this before every merge.

In [ ]:
l = pd.DataFrame({'k': [1, 1, 2], 'a': ['x', 'y', 'z']})
r = pd.DataFrame({'k': [1, 1, 2], 'b': ['p', 'q', 'r']})
print('3 rows x 3 rows, but key 1 appears twice on each side:')
display(pd.merge(l, r, on='k'))
print('result has', len(pd.merge(l, r, on='k')), 'rows, not 3')
print('\ncheck first:  l["k"].is_unique =', l['k'].is_unique)

### Tasks — Lab 11

In [ ]:
orders = pd.DataFrame({
    'order_id'   : [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 1, 3, 5],
    'amount'     : [2500, 1800, 3200, 900, 4100],
})
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'cust_name'  : ['Asel', 'Bek', 'Chyn', 'Dina'],
    'city'       : ['Bishkek', 'Osh', 'Naryn', 'Osh'],
})

# TASK 11.1  Join so that every ORDER keeps its row, with customer details attached.
m1 = ...

# TASK 11.2  Join so that every CUSTOMER appears, even those with no orders.
m2 = ...

# TASK 11.3  Using the result of 11.1, find total order amount per city.
m3 = ...

print('11.1'); display(m1)
print('11.2'); display(m2)
print('11.3'); display(m3)

<details>
<summary><b>Solutions — Lab 11</b></summary>

```python
m1 = pd.merge(orders, customers, on='customer_id', how='left')
m2 = pd.merge(customers, orders, on='customer_id', how='left')
m3 = m1.groupby('city')['amount'].sum()
```
</details>

---

# Lab 12 — Time Series, File I/O and Plotting

### Aim
To work with dates, read and write files, and produce quick plots.

### Theory
`pd.to_datetime` converts strings to `datetime64[ns]`. Once a DatetimeIndex is in place:

- `.dt` accessor gives `.year`, `.month`, `.day_name()`, `.quarter`…
- `resample('ME')` groups by time period (`D` day, `W` week, `ME` month-end, `QE`, `YE`)
- `rolling(n)` computes moving windows
- slicing accepts partial dates: `df['2025-03']`

`df.plot()` wraps matplotlib and works directly inside Colab.

In [ ]:
rng = np.random.default_rng(11)
dates = pd.date_range('2025-01-01', periods=120, freq='D')
ts = pd.DataFrame({
    'date'   : dates,
    'sales'  : rng.integers(200, 900, size=120) + np.arange(120) * 2,  # upward trend
    'visits' : rng.integers(50, 300, size=120),
})
ts['date'] = pd.to_datetime(ts['date'])
ts.head()

In [ ]:
# --- the .dt accessor ---
ts['year']    = ts['date'].dt.year
ts['month']   = ts['date'].dt.month
ts['weekday'] = ts['date'].dt.day_name()
ts['is_weekend'] = ts['date'].dt.weekday >= 5
display(ts.head())

print('\naverage sales by weekday:')
display(ts.groupby('weekday')['sales'].mean().round(0).sort_values(ascending=False))
print('\nweekend vs weekday:')
display(ts.groupby('is_weekend')['sales'].mean().round(0))

In [ ]:
# --- DatetimeIndex, resampling and rolling windows ---
tsi = ts.set_index('date')[['sales', 'visits']]

print('March only (partial-string slice; use .loc):')
display(tsi.loc['2025-03'].head(3))

print('\nmonthly totals:')
display(tsi.resample('ME').sum())

print('\nweekly mean, first 4 weeks:')
display(tsi.resample('W').mean().head(4).round(1))

tsi['sales_7day_avg'] = tsi['sales'].rolling(window=7).mean()
tsi['sales_change']   = tsi['sales'].diff()
tsi['pct_change']     = tsi['sales'].pct_change().round(3)
display(tsi.head(10))

In [ ]:
# --- plotting ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

tsi[['sales', 'sales_7day_avg']].plot(ax=axes[0, 0], title='Daily sales & 7-day average')
tsi['sales'].resample('ME').sum().plot(kind='bar', ax=axes[0, 1], title='Monthly total sales')
tsi['sales'].plot(kind='hist', bins=25, ax=axes[1, 0], title='Distribution of daily sales')
tsi.plot(kind='scatter', x='visits', y='sales', alpha=0.6, ax=axes[1, 1],
         title='Sales vs visits')

plt.tight_layout()
plt.show()

### File I/O

In Colab, files you write land in the temporary `/content` folder — visible in the
**Files** panel on the left. They disappear when the runtime restarts, so download
anything you need to keep.

Common readers: `read_csv`, `read_excel`, `read_json`, `read_html`, `read_parquet`,
`read_sql`. Each has a matching `to_*` writer.

In [ ]:
# --- write, then read back ---
ts.to_csv('sales_data.csv', index=False)
print('written: sales_data.csv')

back = pd.read_csv('sales_data.csv', parse_dates=['date'])
print('read back ->', back.shape, '| date dtype:', back['date'].dtype)
display(back.head(3))

# other formats
ts.head(20).to_json('sample.json', orient='records', date_format='iso')
ts.to_excel('sales_data.xlsx', index=False)   # needs openpyxl, pre-installed in Colab
print('\nalso written: sample.json, sales_data.xlsx')

In [ ]:
# --- useful read_csv options (run this cell to see them in action) ---
demo = pd.read_csv('sales_data.csv',
                   usecols=['date', 'sales', 'visits'],  # only these columns
                   nrows=5,                              # only the first 5 rows
                   parse_dates=['date'])
display(demo)

# reading straight from a URL works too, e.g.:
# url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
# titanic = pd.read_csv(url)

**Uploading your own file in Colab** — run the cell below and pick a file from your
computer. (Skip it if you have nothing to upload; it will simply wait for a selection.)

In [ ]:
# from google.colab import files
# uploaded = files.upload()               # opens a file picker
# my_df = pd.read_csv(next(iter(uploaded)))
# my_df.head()

# To download a file you created:
# files.download('sales_data.csv')

# To mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# pd.read_csv('/content/drive/MyDrive/your_folder/your_file.csv')
print('Uncomment the lines you need.')

### Tasks — Lab 12

In [ ]:
# TASK 12.1  Add a column `quarter` to `ts` using the .dt accessor.
...

# TASK 12.2  Total sales per month, as a Series (use resample on tsi).
q2 = ...

# TASK 12.3  The date on which sales were highest.
q3 = ...

# TASK 12.4  A 30-day rolling maximum of visits; plot it against daily visits.
...

print('12.2'); display(q2)
print('12.3 ->', q3)

<details>
<summary><b>Solutions — Lab 12</b></summary>

```python
ts['quarter'] = ts['date'].dt.quarter
q2 = tsi['sales'].resample('ME').sum()
q3 = tsi['sales'].idxmax()
tsi['visits_30d_max'] = tsi['visits'].rolling(30).max()
tsi[['visits', 'visits_30d_max']].plot(figsize=(10, 4))
```
</details>

---

# Lab 13 — Mini Project: End-to-End Analysis

### Aim
To apply the whole toolkit to one messy dataset and report findings.

### Scenario
You are given one year of transaction records from a small electronics retailer with
three branches. The data has missing values, inconsistent text and wrong dtypes —
as real data always does.

### Required deliverables
Complete the eight steps below. For each, write the code **and** a one-line conclusion
in the markdown cell at the end.

In [ ]:
# --- generate the raw dataset (run this cell as-is) ---
rng = np.random.default_rng(2025)
n = 500

shop = pd.DataFrame({
    'txn_id'  : np.arange(1001, 1001 + n),
    'date'    : rng.choice(pd.date_range('2025-01-01', '2025-12-31'), n).astype(str),
    'branch'  : rng.choice(['bishkek', 'OSH', 'Karakol ', 'Bishkek'], n),
    'category': rng.choice(['Laptop', 'Phone', 'Accessory', 'Tablet'], n,
                           p=[0.25, 0.35, 0.30, 0.10]),
    'units'   : rng.integers(1, 6, n),
    'price'   : rng.choice([48000, 32000, 1500, 21000, 850, 15000], n).astype(float),
    'payment' : rng.choice(['cash', 'card', 'transfer', None], n, p=[.4, .4, .15, .05]),
})

# inject some mess
shop.loc[rng.choice(n, 30, replace=False), 'price'] = np.nan
shop.loc[rng.choice(n, 15, replace=False), 'units'] = np.nan
shop = pd.concat([shop, shop.sample(10, random_state=1)], ignore_index=True)  # duplicates

print('raw shape:', shop.shape)
shop.head()

In [ ]:
# STEP 1  Inspect: shape, dtypes, missing counts, duplicate count.
...

In [ ]:
# STEP 2  Clean:
#   - drop duplicate rows
#   - parse `date` as datetime
#   - normalise `branch` (strip whitespace, title case)
#   - fill missing `price` with the median price OF ITS CATEGORY (hint: groupby + transform)
#   - fill missing `units` with 1
#   - fill missing `payment` with 'unknown'
...

In [ ]:
# STEP 3  Feature engineering:
#   - `revenue` = units * price
#   - `month` (period or month number) and `weekday` name from `date`
#   - `basket_size` band using pd.cut on revenue: small / medium / large
...

In [ ]:
# STEP 4  Which branch earned the most total revenue? Which has the highest average
#         revenue per transaction? Are they the same branch?
...

In [ ]:
# STEP 5  Build a pivot table of total revenue with rows=month, columns=category.
...

In [ ]:
# STEP 6  For each branch, find its single best-selling category by units sold.
...

In [ ]:
# STEP 7  Plot: (a) monthly revenue as a line, (b) revenue share by category as a bar chart.
...

In [ ]:
# STEP 8  Export the cleaned dataset to 'shop_clean.csv' and the month x category
#         pivot to 'summary.xlsx'.
...

### Mini-project report
Double-click to edit and write your findings.

**Dataset:** ______ rows × ______ columns after cleaning (______ duplicates removed,
______ missing values filled).

**Finding 1 — Branch performance:** 

**Finding 2 — Category mix:** 

**Finding 3 — Seasonality:** 

**Finding 4 — Payment behaviour:** 

**Conclusion:** 

---

# Appendix A — Quick Reference

### NumPy

```python
np.array(lst)            np.arange(a, b, s)      np.linspace(a, b, n)
np.zeros(sh)             np.ones(sh)             np.full(sh, v)      np.eye(n)
rng = np.random.default_rng(seed)
rng.random(sh)           rng.integers(lo, hi, sh)  rng.normal(mu, sd, sh)

a.shape  a.ndim  a.size  a.dtype  a.nbytes
a.reshape(r, c)   a.ravel()   a.T   a.astype(t)   a.copy()
a[i, j]  a[r0:r1, c0:c1]  a[mask]  a[[i1, i2]]

a.sum(axis=)  a.mean()  a.std()  a.min()  a.max()  a.cumsum()
a.argmin()  a.argmax()  a.argsort()  np.sort(a)  np.unique(a)
np.where(cond, x, y)   np.clip(a, lo, hi)   np.concatenate([a, b])
a @ b   np.linalg.inv(A)   np.linalg.det(A)   np.linalg.solve(A, b)
```

### pandas

```python
pd.Series(data, index=)          pd.DataFrame(dict_of_lists)
pd.read_csv(path)               df.to_csv(path, index=False)

df.shape  df.head()  df.info()  df.describe()  df.dtypes  df.columns
df.isna().sum()   df['c'].value_counts()   df['c'].unique()

df['c']            df[['c1','c2']]
df.loc[rows, cols]            # by label, end INCLUSIVE
df.iloc[rows, cols]           # by position, end exclusive
df[(df.a > 1) & (df.b == 'x')]        df.query('a > 1')
df.sort_values('c', ascending=False)  df.nlargest(n, 'c')

df.dropna(subset=)  df.fillna(v)  df.drop_duplicates()  df.rename(columns=)
pd.to_numeric(s, errors='coerce')     pd.to_datetime(s)
df['c'].str.strip().str.lower()       df['c'].astype(t)

df.groupby('k')['v'].sum()
df.groupby('k').agg(new=('v', 'mean'))
df.groupby('k')['v'].transform('sum')
pd.pivot_table(df, values=, index=, columns=, aggfunc=)
pd.crosstab(df.a, df.b)               pd.cut(s, bins=, labels=)

pd.merge(l, r, on='k', how='left')    pd.concat([a, b], ignore_index=True)
df.set_index('c')   df.reset_index()  df.apply(f)  df['c'].map(f)
s.dt.year  s.dt.month  s.dt.day_name()   df.resample('ME').sum()
df.plot()   df.plot(kind='bar'|'hist'|'scatter'|'box')
```

---

# Appendix B — Common Errors and Their Cause

| Message | Usual cause | Fix |
|---|---|---|
| `ValueError: operands could not be broadcast together` | incompatible shapes | check `.shape`; add an axis with `[:, None]` |
| `IndexError: index 5 is out of bounds for axis 0 with size 5` | off-by-one | valid indices are `0 .. size-1` |
| `KeyError: 'col'` | column name typo or stray whitespace | `df.columns.tolist()`; `df.columns.str.strip()` |
| `ValueError: The truth value of a Series is ambiguous` | used `and` / `or` on a Series | use `&` / `\|` with parentheses |
| `SettingWithCopyWarning` | chained assignment | one `df.loc[mask, 'col'] = v` |
| `TypeError: unsupported operand type(s) for +: 'int' and 'str'` | numeric column stored as text | `pd.to_numeric(..., errors='coerce')` |
| `ValueError: cannot reshape array of size 12 into shape (5,3)` | sizes must match | product of new shape must equal `size` |
| `NameError: name 'np' is not defined` | ran a cell out of order | `Runtime → Restart and run all` |

---

# Appendix C — Marking Scheme

| Component | Marks |
|---|---:|
| Labs 1–5 tasks (NumPy) | 20 |
| Labs 6–12 tasks (pandas) | 30 |
| Viva answers | 10 |
| Mini project — cleaning & correctness | 20 |
| Mini project — analysis & plots | 10 |
| Report, neatness, reproducibility | 10 |
| **Total** | **100** |

---

*End of lab manual.*